# مشروع 2: تحليل تسرب العملاء (Telco Customer Churn) + Dashboard
- **رابط مستودع GitHub للـ Dashboard:** 
- **رابط تطبيق Streamlit المنشور:** 

## 1. فهم المشكلة (Problem Understanding)
- **ما السؤال؟** ما الأسباب التي تجعل عملاء شركة الاتصالات يتركون الشركة (Churn)، ومن هم العملاء الأكثر احتمالاً للمغادرة؟
- **ما الـ Target؟** عمود `Churn` (قيمته `Yes` إذا غادر العميل، و `No` إذا استمر).
- **من سيستفيد من الإجابة؟** فريق التسويق وخدمة العملاء والإدارة، لتقديم عروض مناسبة للعملاء المعرضين للمغادرة قبل ترك الخدمة.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 2. أول نظرة على البيانات (First Look)
نستعرض حجم البيانات، أنواع الأعمدة، القيم الناقصة، التكرارات، والإحصائيات الأساسية.

In [ ]:
print("حجم البيانات:", df.shape)
print("\n--- معلومات الأعمدة ---")
df.info()

print("\n--- القيم المفقودة ---")
print(df.isnull().sum())

print("\n--- الصفوف المكررة ---")
print(df.duplicated().sum())

display(df.describe())
display(df.describe(include='object'))

**ملاحظات أول نظرة (إجابة أسئلة التفكير):**
- البيانات تتكون من **7,043** صفاً و **21** عموداً، ولا يوجد أي صف مكرر (`0`).
- **هل كل عمود رقمي مخزن فعلاً كرقم؟** لا، عمود `TotalCharges` مخزن كـ `object` (نص) وليس رقمي بسبب وجود مسافات فارغة `" "` للعملاء الجدد.
- **هل كل عمود Yes / No مكتوب بنفس الطريقة؟** نعم، كلها مكتوبة بحروف كبيرة في البداية (`Yes` و `No`)، ما عدا عمود `SeniorCitizen` فهو مخزن كـ `0` و `1`.
- **الأعمدة ذات الثلاث قيم:** أعمدة الخدمات الإضافية مثل `OnlineSecurity` و `MultipleLines` تحتوي على قيمة ثالثة (`No internet service` أو `No phone service`). هذه القيمة ليست معلومة جديدة بل هي تكرار لحالة العمود الأساسي (`InternetService == 'No'` أو `PhoneService == 'No'`).

## 3. الاستكشاف بالرسومات (EDA)
نستكشف توزيع عمود الهدف (`Churn`) وعلاقته بأهم الأعمدة مثل نوع العقد (`Contract`) ومدة الاشتراك (`tenure`).

In [ ]:
# توزيع عمود الهدف Churn
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Churn', palette='pastel')
plt.title('توزيع العملاء حسب المغادرة (Churn)')
plt.xlabel('هل غادر العميل؟')
plt.ylabel('عدد العملاء')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()), ha='center', va='bottom')
plt.show()

نلاحظ من الرسم أن **1,869** عميلاً (بنسبة **26.5%**) غادروا الشركة مقابل **5,174** عميلاً استمروا معها.

In [ ]:
# العلاقة بين نوع العقد (Contract) والمغادرة
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2')
plt.title('المغادرة حسب نوع العقد')
plt.xlabel('نوع العقد')
plt.ylabel('عدد العملاء')
plt.show()

print(pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100)

العملاء أصحاب العقود الشهرية (`Month-to-month`) هم الأكثر مغادرة بنسبة **42.7%**، بينما تنخفض النسبة إلى **2.8%** فقط في عقود السنتين.

In [ ]:
# توزيع مدة الاشتراك (tenure) حسب المغادرة
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Churn', y='tenure', palette='coolwarm')
plt.title('مدة الاشتراك بالأشهر مقارنة بالمغادرة')
plt.xlabel('هل غادر العميل؟')
plt.ylabel('مدة الاشتراك (بالأشهر)')
plt.show()

print(df.groupby('Churn')['tenure'].median())

وسيط مدة اشتراك العملاء المغادرين هو **10 أشهر** فقط، مقارنة بـ **38 شهراً** للعملاء المستمرين.

## 4. تنظيف البيانات (Data Cleaning + Decision Log)
ننظف البيانات ونوثق كل قرار مع سببه وعدد الصفوف المتأثرة.

In [ ]:
df_clean = df.copy()

# 1. تحويل TotalCharges إلى أرقام ومعالجة المسافات الفارغة
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
missing_tc = df_clean['TotalCharges'].isnull().sum()
print("عدد الصفوف الفارغة في TotalCharges:", missing_tc)

df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)

# 2. توحيد القيم المكررة في أعمدة الخدمات إلى No
cols_to_fix = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in cols_to_fix:
    count_replaced = df_clean[col].isin(['No internet service', 'No phone service']).sum()
    df_clean[col] = df_clean[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    print(f"العمود {col}: تم توحيد {count_replaced} صفاً إلى No")

# 3. حذف عمود customerID
df_clean.drop(columns=['customerID'], inplace=True)

# حفظ البيانات النظيفة
df_clean.to_csv('telco_clean.csv', index=False)
print("\nتم حفظ ملف telco_clean.csv بنجاح، الحجم الجديد:", df_clean.shape)

### سجل القرارات (Decision Log):
| القرار | السبب | عدد الصفوف المتأثرة |
| :--- | :--- | :--- |
| تحويل `TotalCharges` إلى أرقام وتعويض الفراغات بـ `0` | كانت مخزنة كنصوص بسبب مسافات فارغة لعملاء جدد مدة اشتراكهم `0` شهر | **11 صفاً** |
| استبدال `No internet service` بـ `No` في 6 أعمدة خدمات | المعلومة مكررة وموجودة مسبقاً في عمود `InternetService` | **1,526 صفاً** في كل عمود |
| استبدال `No phone service` بـ `No` في عمود `MultipleLines` | المعلومة مكررة وموجودة مسبقاً في عمود `PhoneService` | **682 صفاً** |
| حذف عمود `customerID` | معرف نصي فريد لكل عميل لا يفيد في التحليل أو الموديل | **7,043 صفاً** (حذف عمود) |

## 5. هندسة الميزات (Feature Engineering)
ننشئ أعمدة جديدة تعطي معنى أوضح لسلوك العميل:
- `TenureGroup`: تقسيم مدة الاشتراك إلى فئات (جديد، متوسط، قديم).
- `TotalServices`: حساب عدد الخدمات الإضافية المشترك بها العميل.
- `AvgMonthlySpend`: متوسط الإنفاق الشهري الفعلي.

In [ ]:
df_fe = df_clean.copy()

# 1. تقسيم مدة الاشتراك إلى فئات
def group_tenure(t):
    if t <= 12:
        return 'New (0-1 Year)'
    elif t <= 36:
        return 'Mid (1-3 Years)'
    else:
        return 'Loyal (3+ Years)'

df_fe['TenureGroup'] = df_fe['tenure'].apply(group_tenure)

# 2. حساب عدد الخدمات الإضافية
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_fe['TotalServices'] = (df_fe[service_cols] == 'Yes').sum(axis=1)

# 3. متوسط الإنفاق الشهري الفعلي
df_fe['AvgMonthlySpend'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# رسمة توضح علاقة فئة مدة الاشتراك بالمغادرة
plt.figure(figsize=(8, 5))
sns.countplot(data=df_fe, x='TenureGroup', hue='Churn', order=['New (0-1 Year)', 'Mid (1-3 Years)', 'Loyal (3+ Years)'], palette='Set1')
plt.title('المغادرة حسب فئة مدة الاشتراك (TenureGroup)')
plt.xlabel('فئة مدة الاشتراك')
plt.ylabel('عدد العملاء')
plt.show()

print(pd.crosstab(df_fe['TenureGroup'], df_fe['Churn'], normalize='index') * 100)

يوضح الرسم أن العملاء الجدد في أول سنة (`New 0-1 Year`) تصل نسبة مغادرتهم إلى **47.4%**، بينما تنخفض لدى العملاء القدامى (`3+ Years`) إلى **11.9%** فقط.

## 6. اختيار الميزات (Feature Selection)
نحول الأعمدة الفئوية إلى أرقام ونفحص الارتباط مع عمود الهدف `Churn` لاختيار أهم الميزات وحفظ ملف `telco_model_ready.csv`.

In [ ]:
df_model = df_fe.copy()
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

df_encoded = pd.get_dummies(df_model, drop_first=True, dtype=int)

correlations = df_encoded.corr()['Churn'].sort_values(ascending=False)
print("أعلى 5 ميزات ارتباطاً إيجابياً مع Churn:")
print(correlations.drop('Churn').head(5))
print("\nأعلى 5 ميزات ارتباطاً سلبياً مع Churn:")
print(correlations.tail(5))

top_corr = pd.concat([correlations.drop('Churn').head(5), correlations.tail(5)])
plt.figure(figsize=(9, 5))
top_corr.plot(kind='barh', color=['salmon' if x > 0 else 'skyblue' for x in top_corr])
plt.title('أقوى 10 ميزات مرتبطة بمغادرة العملاء (Churn)')
plt.xlabel('معامل الارتباط (Correlation)')
plt.show()

# حذف عمود gender لضعف ارتباطه الشديد
cols_to_drop = [col for col in df_encoded.columns if 'gender_' in col]
df_model_ready = df_encoded.drop(columns=cols_to_drop)

df_model_ready.to_csv('telco_model_ready.csv', index=False)
print("\nتم حفظ ملف telco_model_ready.csv بنجاح، الحجم النهائي:", df_model_ready.shape)

يُظهر الرسم أن استخدام إنترنت الألياف الضوئية (`Fiber optic`) والدفع بالشيك الإلكتروني يرتبطان إيجابياً بالمغادرة (بمعامل ارتباط **+0.31** و **+0.30**)، بينما طول مدة الاشتراك (`tenure`) يقلل المغادرة بمعامل ارتباط **-0.35**.

### جدول قرار اختيار الميزات (Feature Selection Decision Table):
| الميزة / المجموعة | القرار | الدليل / السبب |
| :--- | :--- | :--- |
| `tenure` و `TenureGroup` | **إبقاء** | أقوى ارتباط سلبي مع المغادرة (`-0.35`) |
| `Contract` | **إبقاء** | فرق كبير في نسبة المغادرة بين العقد الشهري (`42.7%`) وعقد السنتين (`2.8%`) |
| `InternetService` | **إبقاء** | عملاء Fiber optic لديهم ارتباط إيجابي عالٍ بالمغادرة (`+0.31`) |
| `MonthlyCharges` و `TotalServices` | **إبقاء** | ارتفاع الفاتورة الشهرية يزيد من احتمالية ترك الخدمة |
| `gender` | **حذف** | معامل الارتباط مع `Churn` يقارب الصفر (`-0.008`) ولا يؤثر على القرار |

## 7. النتائج والتوصيات (Results & Recommendations)
**أهم ما اكتشفناه:**
1. نسبة التسرب الإجمالية في الشركة تبلغ **26.5%** (1,869 عميلاً من أصل 7,043).
2. العقود الشهرية (`Month-to-month`) هي الخطر الأكبر بنسبة مغادرة **42.7%**.
3. أول سنة هي الفترة الحرجة؛ حيث يغادر **47.4%** من العملاء الجدد خلال أول 12 شهراً.

**التوصيات:**
- **لفريق التسويق والمبيعات:** تقديم خصم أو شهر مجاني للعملاء أصحاب العقود الشهرية لتحويلهم إلى عقود سنوية أو لمدة سنتين.
- **لفريق خدمة العملاء:** التواصل الاستباقي مع العملاء الجدد خلال أول **6 إلى 12 شهراً** للتأكد من رضاهم عن الخدمة، خاصة مشتركي الألياف الضوئية (`Fiber optic`).

## 8. كود لوحة التحكم (app.py) وملف المكتبات (requirements.txt)
هنا كود لوحة Streamlit وملف `requirements.txt` مدمجين داخل النوتبوك، ويمكن تشغيل الخلية التالية لتوليدهما تلقائياً في نفس المجلد.

In [ ]:
app_code = """" + app_code_str + """"

req_code = """" + req_str + """"

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip() + "\n")

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(req_code.strip() + "\n")

print("تم إنشاء app.py و requirements.txt بنجاح.")

## 9. تقرير الأسبوع (Weekly Report)

خلال هذا الأسبوع تعلمت كيف أمشي في مشروع تحليل بيانات متكامل من البداية حتى النهاية، بدءاً من فهم المشكلة واستكشاف البيانات وصولاً إلى تنظيفها وتجهيزها وبناء لوحة تحكم تفاعلية. عند مقارنة المشروعين، كانت المشكلة الأصعب في بيانات السكري هي التعامل مع القيم الصفرية غير المنطقية (مثل صفر في قياس الجلوكوز أو ضغط الدم) والتي كانت تخفي قيماً مفقودة تحتاج إلى تعويض مدروس، أما في بيانات عملاء الاتصالات فكانت الصعوبة في كثرة الأعمدة النصية والفئوية ووجود مسافات فارغة مخفية داخل عمود رقمي (`TotalCharges`) بالإضافة إلى تكرار بعض القيم في أعمدة الخدمات. بالنسبة لي كانت بيانات عملاء الاتصالات أصعب قليلاً لأنها تتطلب قرارات أكثر في توحيد النصوص وتحويلها إلى أرقام قبل الموديل. أفضل رسمة رسمتها هذا الأسبوع هي رسمة العلاقة بين نوع العقد (`Contract`) ونسبة المغادرة (`Churn`)، لأنها كشفت المشكلة بوضوح شديد في نظرة واحدة وأظهرت الفرق الكبير بين العقد الشهري وعقد السنتين. الآن أصبحت أسرع بكثير في كتابة أكواد الاستكشاف والتنظيف باستخدام `Pandas` ورسم المخططات بـ `Seaborn` مقارنة بأول يوم في الأسبوع حيث كنت أبحث عن أبسط الدوال، لكن ما زال اختيار أفضل طريقة لهندسة الميزات وتحديد أي الأعمدة نحذفها في مرحلة اختيار الميزات يحتاج مني مزيداً من الممارسة. بناءً على تحليلي لبيانات الشركة، أوصي إدارة التسويق بالتركيز على تحويل عملاء العقود الشهرية إلى عقود سنوية عبر عروض تحفيزية، لأن **42.7%** من أصحاب العقود الشهرية يتركون الشركة مقارنة بـ **2.8%** فقط في عقود السنتين.